# AMB Kernel · ShellForce 학습과 시각화

독립 dataset 생성/재사용 → 학습 → best checkpoint 복원 → test 시각화를 실행합니다.
학습 후 **profile video, EP map video, time averaged EP map, EP spectrum**을 확인합니다.

파라미터 sweep, control 비교, 적분·stationarity 검증은
[Sanity_AMB_Kernel.ipynb](Sanity_AMB_Kernel.ipynb)에서 수행합니다.
[모델 및 실행 안내](../data/amb_kernel/README.md)

In [ ]:
from pathlib import Path
from dataclasses import asdict
from datetime import datetime, timezone, timedelta
from time import perf_counter
import json
import sys
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents, HERE / 'CNEEP_v2']
             if (p / 'models/NEEP_ShellForce_2D.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run Jupyter inside CNEEP_v2 or its parent.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image
from tqdm.auto import tqdm
from data.amb_kernel.model import AMBKernelModel, ModelConfig, MODEL_NAMES, pilot_config
from data.amb_kernel.experiment import RunConfig
from data.amb_kernel.dataset import DatasetConfig, generate_dataset, resolve_simulation_device
from data.amb_kernel.learning import ArchitectureConfig, TrainingConfig, train_shell_force, load_checkpoint, source_provenance
from data.amb_kernel.visualization import analyze_test_fields
from data.amb_kernel.plotting import render_training_visualizations
NOTEBOOK_STARTED = perf_counter()
STAGE_TIMINGS = []
SHOW_PROGRESS = True
BAR_FORMAT = '{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed} elapsed, ETA {remaining}, {rate_fmt}{postfix}]'
def record_stage(name, started):
    elapsed = perf_counter() - started
    STAGE_TIMINGS.append({'stage': name, 'elapsed_seconds': elapsed})
    tqdm.write(f'{name}: {timedelta(seconds=round(elapsed))} ({elapsed:.1f} s)')
torch.set_num_threads(min(4, torch.get_num_threads()))
print('Repository:', ROOT, '| Torch:', torch.__version__)

## 1. 학습 설정

MODEL_IDS에 학습할 모델을 지정하고 MODEL_OVERRIDES에 모델별 하나의 물리 설정을 지정합니다.
dt=0.01, burn-in 10,000 steps, 측정 1,000 intervals이며 물리 시간은 각각 100, 10입니다.
Train/validation/test는 1,024/256/10개의 독립 trajectory, trajectory당 1,001 frame입니다.
Optimizer 업데이트 횟수는 TRAINING.iterations입니다.

학습 grid는 32×32이며 t=0의 초기조건은 평균을 보존하는 circle profile입니다.
INITIAL_MODE, INITIAL_NOISE_SCALE, CIRCLE_RADIUS_FRACTION, CIRCLE_WIDTH, CIRCLE_AMPLITUDE에서 조절합니다.
Circle은 초기조건이며 물리 parameter에 따라 burn-in 동안 완화될 수 있습니다.
모델의 a/b 등 물리 계수는 MODEL_OVERRIDES에서 명시한 값으로 유지합니다.

SIMULATION_BATCH_SIZE는 GPU에서 함께 적분할 trajectory 수입니다.
SIMULATION_DEVICE='cuda'는 CUDA가 없으면 오류를 내며, 'auto'는 사용 가능한 장치를 선택합니다.
DATASET_ROOT에 모델별 폴더를 포함한 기존 dataset 경로를 지정하면 호환되는 완료 자료를 재사용합니다.

In [ ]:
MODEL_IDS = ('M00', 'M01', 'M02', 'M03', 'M04', 'M05')
MODEL_OVERRIDES = {model_id: {} for model_id in MODEL_IDS}
TRAIN_M, VAL_M, TEST_M = 1024, 256, 10
N_INTERVALS = 1000
BURN_IN_STEPS = 10000
TRAIN_N = 32
TRAIN_DT = 0.01
TRAIN_OBSERVATION_STRIDE = 1
INITIAL_MODE = 'circle'
INITIAL_NOISE_SCALE = 0.0
CIRCLE_RADIUS_FRACTION = 0.25
CIRCLE_WIDTH = 1.0
CIRCLE_AMPLITUDE = 1.0
SIMULATION_DEVICE = 'auto'
SIMULATION_BATCH_SIZE = 32
SPLIT_SEEDS = (12031, 23041, 34057)
DATASET_ROOT = None
ARCHITECTURE = ArchitectureConfig(n_channel=64, n_hidden=3, max_distance=3,
                                  shell_force_activation='elu')
TRAINING = TrainingConfig(iterations=2000, batch_size=2048, eval_batch_size=2048,
                          micro_batch_size=32, learning_rate=1e-3,
                          alpha=-0.5, record_every=100, seed=3,
                          device='cuda' if torch.cuda.is_available() else 'cpu',
                          dtype='float64', lag_frames=1)
VISUALIZATION_BATCH_SIZE = 32
VIDEO_TRAJECTORY = 0
VIDEO_FRAME_STRIDE = 5
VIDEO_MAX_FRAMES = 200
VIDEO_FPS = 10

DATASET = DatasetConfig(train_count=TRAIN_M, validation_count=VAL_M, test_count=TEST_M,
                       intervals=N_INTERVALS, burn_in_steps=BURN_IN_STEPS,
                       observation_stride=TRAIN_OBSERVATION_STRIDE, split_seeds=SPLIT_SEEDS)
DATA_RUN = RunConfig(dt=TRAIN_DT, burn_in_steps=DATASET.burn_in_steps,
                    measurement_steps=DATASET.intervals * DATASET.observation_stride,
                    observation_stride=DATASET.observation_stride,
                    observation_lags=tuple(DATASET.observation_stride * k for k in (1, 4, 16)),
                    block_steps=min(256, DATASET.intervals * DATASET.observation_stride),
                    max_distance=ARCHITECTURE.max_distance,
                    initial_mode=INITIAL_MODE, initial_scale=INITIAL_NOISE_SCALE,
                    circle_radius_fraction=CIRCLE_RADIUS_FRACTION,
                    circle_width=CIRCLE_WIDTH, circle_amplitude=CIRCLE_AMPLITUDE)
NN_METADATA = vars(ARCHITECTURE.options())
resolved_simulation_device = resolve_simulation_device(SIMULATION_DEVICE)
if not MODEL_IDS or any(mid not in MODEL_NAMES for mid in MODEL_IDS) or len(set(MODEL_IDS)) != len(MODEL_IDS):
    raise ValueError('MODEL_IDS must contain distinct supported model IDs.')
if TRAINING.lag_frames > N_INTERVALS:
    raise ValueError('Training lag exceeds the trajectory length.')
if isinstance(SIMULATION_BATCH_SIZE, bool) or not isinstance(SIMULATION_BATCH_SIZE, int) or SIMULATION_BATCH_SIZE < 1:
    raise ValueError('SIMULATION_BATCH_SIZE must be a positive integer.')
stamp = datetime.now(timezone(timedelta(hours=9))).strftime('%Y%m%d-%H%M%S-%f')
OUTPUT = ROOT / 'results' / 'amb_kernel' / 'training' / stamp
OUTPUT.mkdir(parents=True, exist_ok=False)
dataset_root = Path(DATASET_ROOT) if DATASET_ROOT is not None else OUTPUT / 'datasets'
frame_dt = DATA_RUN.dt * DATASET.observation_stride
display(pd.DataFrame([{'split': name, 'trajectories': count,
                       'frames_per_trajectory': DATASET.frames_per_trajectory,
                       'adjacent_pairs': count * DATASET.intervals}
                      for name, count in DATASET.counts.items()]))
print('Models:', ', '.join(MODEL_IDS), '| simulation:', resolved_simulation_device, '| training:', TRAINING.device)
print('Parallel trajectories:', min(SIMULATION_BATCH_SIZE, DATASET.total_replicas)
      if resolved_simulation_device.startswith('cuda') else 1)
print(f'Physical time: burn-in {BURN_IN_STEPS * TRAIN_DT:g}, measurement {N_INTERVALS * frame_dt:g}.')
print(f'Field storage: {DATASET.estimated_storage_bytes(TRAIN_N) / 2**30:.2f} GiB/model.')
print('Output:', OUTPUT)
(OUTPUT / 'training_plan.json').write_text(json.dumps({
    'model_ids': MODEL_IDS, 'physical_overrides': MODEL_OVERRIDES,
    'dataset_config': asdict(DATASET), 'grid_n': TRAIN_N, 'run_config': asdict(DATA_RUN),
    'simulation_device_requested': SIMULATION_DEVICE, 'simulation_device_resolved': resolved_simulation_device,
    'simulation_batch_size': SIMULATION_BATCH_SIZE,
    'training_config': asdict(TRAINING), 'architecture': vars(ARCHITECTURE.options()),
    'dataset_root': str(dataset_root), 'source_provenance': source_provenance(),
    'video': {'trajectory': VIDEO_TRAJECTORY, 'stride': VIDEO_FRAME_STRIDE,
              'max_frames': VIDEO_MAX_FRAMES, 'fps': VIDEO_FPS},
}, indent=2), encoding='utf8')

### 초기 profile 확인 · t=0, burn-in 이전

각 모델의 첫 train replica와 같은 seed·초기조건 생성 함수를 사용한 실제 초기 field입니다.
질량 평균은 phi_bar에 맞춥니다. 아래 profile video는 burn-in 이후 저장된 test trajectory이므로
이 초기 그림과 모양이 달라질 수 있습니다.

In [ ]:
initial_preview_seed = DATASET.replica_seeds()['train'][0]
initial_profiles = {}
for model_id in MODEL_IDS:
    physical = AMBKernelModel(pilot_config(model_id, n=TRAIN_N, **MODEL_OVERRIDES[model_id]))
    initial_profiles[model_id] = physical.initial_field(
        np.random.default_rng(initial_preview_seed), scale=DATA_RUN.initial_scale,
        mode=DATA_RUN.initial_mode, circle_radius_fraction=DATA_RUN.circle_radius_fraction,
        circle_width=DATA_RUN.circle_width, circle_amplitude=DATA_RUN.circle_amplitude)

columns = min(3, len(initial_profiles))
rows = int(np.ceil(len(initial_profiles) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.8 * rows), squeeze=False)
vmin = min(float(field.min()) for field in initial_profiles.values())
vmax = max(float(field.max()) for field in initial_profiles.values())
for ax, (model_id, field) in zip(axes.flat, initial_profiles.items()):
    image = ax.imshow(field, origin='lower', cmap='RdBu_r', vmin=vmin, vmax=vmax)
    ax.set(title=f'{model_id}: t=0 before burn-in', xlabel='x', ylabel='y')
    fig.colorbar(image, ax=ax, shrink=.75)
    print(f'{model_id}: initial mean={field.mean():.8g}, shape={field.shape}')
for ax in list(axes.flat)[len(initial_profiles):]:
    ax.axis('off')
fig.tight_layout()
plt.show()

## 2. Dataset 생성 및 학습

Train split에서만 정규화를 구하고 micro batch별로 gradient를 누적합니다.
Validation loss가 가장 낮은 checkpoint를 저장합니다.
한 모델의 학습이 끝나면 dataset memory map과 학습 모델을 해제합니다.

In [ ]:
training_outputs = {}
training_histories = {}
pipeline_started = perf_counter()
for model_id in tqdm(MODEL_IDS, desc='Dataset / training', unit='model',
                     disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
    physical = AMBKernelModel(pilot_config(model_id, n=TRAIN_N, **MODEL_OVERRIDES[model_id]))
    started = perf_counter()
    dataset = generate_dataset(physical, dataset_root / model_id, dataset_config=DATASET,
                               run_config=DATA_RUN, show_progress=SHOW_PROGRESS, resume=True,
                               nn_metadata=NN_METADATA, device=SIMULATION_DEVICE,
                               simulation_batch_size=SIMULATION_BATCH_SIZE)
    record_stage(f'{model_id}/dataset', started)
    try:
        started = perf_counter()
        splits = dataset.training_splits()
        fit = train_shell_force(splits, OUTPUT / 'learning' / model_id,
                                architecture=ARCHITECTURE, config=TRAINING,
                                physical_metadata=dataset.manifest, show_progress=SHOW_PROGRESS,
                                progress_desc=f'{model_id} training')
        record_stage(f'{model_id}/training', started)
        training_histories[model_id] = fit.history
        training_outputs[model_id] = {
            'checkpoint': str(fit.checkpoint), 'dataset_dir': str(dataset.output_dir),
            'test_paths': [str(dataset.output_dir / row['path'] / 'fields.npy')
                           for row in dataset.manifest['replicas'] if row['split'] == 'test'],
            'model_parameters': asdict(dataset.model.config),
            'frame_dt': frame_dt, 'time_offset': DATA_RUN.burn_in_steps * DATA_RUN.dt,
        }
        tqdm.write(f'{model_id}: {fit.checkpoint}')
        del fit, splits
    finally:
        dataset.close()
        del dataset
record_stage('dataset_and_training_all_models', pipeline_started)
(OUTPUT / 'training_outputs.json').write_text(json.dumps(training_outputs, indent=2), encoding='utf8')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for model_id, history in training_histories.items():
    table = pd.DataFrame(history)
    ax.plot(table.iteration, table.validation_loss, label=model_id)
ax.set(xlabel='Optimizer iteration', ylabel='Validation loss')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'validation_loss.png', dpi=150)
plt.show()

## 3. Best checkpoint로 test 결과 계산

EP map은 격자당 rate이며 공간 **합**이 extensive EP rate입니다.
Known map은 같은 관측 pair의 active midpoint work, NN map은 network 추정값입니다.
시간 평균 map과 spectrum은 **모든 test trajectory의 모든 선택 lag pair**를 사용합니다.
Video stride와 최대 frame 수는 표시 frame만 줄이며 EP의 시간 간격을 바꾸지 않습니다.

In [ ]:
visualization_paths = {}
for model_id, entry in tqdm(list(training_outputs.items()), desc='Test maps / videos / spectrum',
                            unit='model', disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
    started = perf_counter()
    fit = load_checkpoint(entry['checkpoint'], device=TRAINING.device)
    with np.load(Path(entry['dataset_dir']) / 'dataset_kernel.npz', allow_pickle=False) as saved:
        physical = AMBKernelModel(ModelConfig(**entry['model_parameters']), kernel=saved['K'])
    test_fields = []
    try:
        for path in entry['test_paths']:
            test_fields.append(np.load(path, mmap_mode='r', allow_pickle=False))
        analysis = analyze_test_fields(
            fit, test_fields, physical, entry['frame_dt'],
            lag_frames=TRAINING.lag_frames, batch_size=VISUALIZATION_BATCH_SIZE,
            video_trajectory=VIDEO_TRAJECTORY, video_stride=VIDEO_FRAME_STRIDE,
            video_max_frames=VIDEO_MAX_FRAMES, time_offset=entry['time_offset'],
            show_progress=SHOW_PROGRESS)
        visualization_paths[model_id] = render_training_visualizations(
            analysis, OUTPUT / 'visualizations' / model_id, model_id=model_id,
            fps=VIDEO_FPS, show_progress=SHOW_PROGRESS)
        display({'model_id': model_id, **analysis['metadata']})
        del analysis
    finally:
        for fields in test_fields:
            fields._mmap.close()
        del fit
    record_stage(f'{model_id}/test_visualizations', started)
(OUTPUT / 'visualization_paths.json').write_text(json.dumps(
    {mid: {key: str(value) for key, value in paths.items()}
     for mid, paths in visualization_paths.items()}, indent=2), encoding='utf8')

## 4. Profile video

선택한 test trajectory의 실제 scalar field이며 시작 frame은 burn-in 이후입니다.
Circle 초기조건은 앞의 t=0 preview에서 확인합니다. VIDEO_FPS는 재생 속도이고 제목에는 물리 시간을 표시합니다.

In [ ]:
for model_id, paths in visualization_paths.items():
    display(HTML(f'<h3>{model_id} · profile</h3>'))
    display(HTML(Path(paths['profile_video_html']).read_text(encoding='utf8')))

## 5. EP map video

Known endpoint work와 NN local EP rate를 같은 색 범위로 비교하며 음의 기여도 표시합니다.

In [ ]:
for model_id, paths in visualization_paths.items():
    display(HTML(f'<h3>{model_id} · EP map</h3>'))
    display(HTML(Path(paths['ep_map_video_html']).read_text(encoding='utf8')))

## 6. Time averaged EP map

전체 test pair에 대한 평균입니다. Video에 표시한 일부 frame의 평균이 아닙니다.

In [ ]:
for model_id, paths in visualization_paths.items():
    print(model_id)
    display(Image(filename=str(paths['time_averaged_ep_map_png'])))

## 7. EP spectrum

Chebyshev shell별 signed extensive EP rate입니다. 물리 shell work와 NN raw branch 분해를 구분합니다.
NN 반경 밖 물리 기여는 tail로 표시하며 NN branch와 물리 mechanism이 동일하다고 가정하지 않습니다.

In [ ]:
for model_id, paths in visualization_paths.items():
    print(model_id)
    display(Image(filename=str(paths['ep_spectrum_png'])))

In [ ]:
timing_table = pd.DataFrame(STAGE_TIMINGS)
display(timing_table)
timing_table.to_csv(OUTPUT / 'stage_timings.csv', index=False)
total_elapsed = perf_counter() - NOTEBOOK_STARTED
(OUTPUT / 'elapsed_summary.json').write_text(json.dumps({
    'notebook_wall_seconds': total_elapsed, 'stages': STAGE_TIMINGS,
    'note': 'Wall time includes pauses between cells; nested timings are not additive.',
}, indent=2), encoding='utf8')
print(f'Total elapsed: {timedelta(seconds=round(total_elapsed))}. Results: {OUTPUT}')